# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SohailAkhtarChanna/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Loading your CSV data into DuckDB

To load your CSV file, first ensure it's uploaded to your Colab environment. You can do this by clicking the 'Files' icon on the left sidebar (looks like a folder) and then clicking the 'Upload to session storage' icon (looks like a page with an arrow pointing up). Upload your `your_february_data.csv` file there.

Once uploaded, you can use the following code to load it into a DuckDB table. Remember to update the `FEB` variable in cell `Fqmrq6tbOEJh` with the actual table name (e.g., `february_data`) after running this cell successfully.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
import duckdb
import os
import pandas as pd
import numpy as np
from google.colab import userdata

# Connect
con = duckdb.connect()

# Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not found in Colab Secrets."

# Hugging Face connection
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

# FlyRank warehouse
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# February = feature window
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Setup complete.")
print("FEB:", FEB)

Setup complete.
FEB: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')


In [22]:
# SECTION 1: Check the two signals

# ---------- SIGNAL 1: SEARCH VOLUME ----------
volume_check = con.sql(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN impressions < 100 THEN '<100'
        WHEN impressions < 500 THEN '100-499'
        WHEN impressions < 1000 THEN '500-999'
        WHEN impressions < 5000 THEN '1k-4.9k'
        ELSE '5k+'
    END AS volume_bucket,
    COUNT(*) AS n,
    ROUND(
        AVG(clicks * 1.0 / NULLIF(impressions, 0)), 4
    ) AS avg_ctr
FROM base
WHERE impressions > 0
GROUP BY 1
ORDER BY
    CASE volume_bucket
        WHEN '<100' THEN 1
        WHEN '100-499' THEN 2
        WHEN '500-999' THEN 3
        WHEN '1k-4.9k' THEN 4
        ELSE 5
    END
""").df()

print("SIGNAL 1 — SEARCH VOLUME")
display(volume_check)

print("VERDICT: CONFIRMED")
print("Search volume is a useful opportunity signal and is linked to the quick-win/volume idea.")


# ---------- SIGNAL 2: CTR VS POSITION ----------
position_check = con.sql(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks,
        SUM(CASE WHEN gsc_data_available THEN gsc_sum_position ELSE 0 END) * 1.0
            / NULLIF(
                SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END),
                0
            ) AS avg_position
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN avg_position <= 3 THEN '1-3'
        WHEN avg_position <= 10 THEN '4-10'
        WHEN avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(
        AVG(clicks * 1.0 / NULLIF(impressions, 0)), 4
    ) AS avg_ctr
FROM base
WHERE impressions > 0
  AND avg_position > 0
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        ELSE 4
    END
""").df()

print("\nSIGNAL 2 — CTR VS POSITION")
display(position_check)

print("VERDICT: CONFIRMED")
print("CTR changes with search position, so position is useful context for the baseline rule.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — SEARCH VOLUME


,volume_bucket,n,avg_ctr
0,<100,73237,0.0074
1,100-499,33188,0.0024
2,500-999,13827,0.0026
3,1k-4.9k,24937,0.0031
4,5k+,8370,0.0033


VERDICT: CONFIRMED
Search volume is a useful opportunity signal and is linked to the quick-win/volume idea.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


SIGNAL 2 — CTR VS POSITION


,position_bucket,n,avg_ctr
0,1-3,18571,0.0086
1,4-10,75602,0.0052
2,11-20,30811,0.0031
3,21+,26972,0.0025


VERDICT: CONFIRMED
CTR changes with search position, so position is useful context for the baseline rule.


### My rule

I will prioritize content that has meaningful search visibility but appears to capture fewer clicks than expected for its search position.

The score gives higher priority to pages with higher search impressions and a larger negative CTR gap relative to their position bucket.

**Reason code:** `high_volume_low_ctr`

**Action label:** `REVIEW_CTR`

This is a simple decision-support baseline, not a causal claim. A high score means the page is worth reviewing first, not that its CTR is definitely incorrect.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 2: Build the ranked baseline queue

queue = con.sql(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks,

        SUM(CASE WHEN gsc_data_available THEN gsc_sum_position ELSE 0 END) * 1.0
            / NULLIF(
                SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END),
                0
            ) AS avg_position

    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
),

with_ctr AS (
    SELECT
        *,
        clicks * 1.0 / NULLIF(impressions, 0) AS ctr
    FROM base
    WHERE impressions >= 100
),

with_bucket AS (
    SELECT
        *,
        CASE
            WHEN avg_position <= 3 THEN '1-3'
            WHEN avg_position <= 10 THEN '4-10'
            WHEN avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket
    FROM with_ctr
    WHERE avg_position > 0
),

bucket_ctr AS (
    SELECT
        position_bucket,
        AVG(ctr) AS bucket_avg_ctr
    FROM with_bucket
    GROUP BY position_bucket
)

SELECT
    w.client_hash_id,
    w.content_hash_id,
    w.impressions,
    w.clicks,
    w.ctr,
    w.avg_position,
    w.position_bucket,
    b.bucket_avg_ctr,
    b.bucket_avg_ctr - w.ctr AS ctr_gap

FROM with_bucket w
JOIN bucket_ctr b
    USING (position_bucket)
""").df()

# Only pages whose CTR is below the average for their position bucket
queue = queue[queue["ctr_gap"] > 0].copy()

# Transparent baseline score
queue["score"] = (
    np.log1p(queue["impressions"]) * queue["ctr_gap"]
)

# One reason code + one action
queue["reason_code"] = "high_volume_low_ctr"
queue["action"] = "REVIEW_CTR"

# Rank highest opportunity first
queue = queue.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Final output columns
queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "position_bucket",
        "bucket_avg_ctr",
        "ctr_gap"
    ]
]

# Write CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Rows in ranked queue:", len(queue))
print("CSV written to:", output_path)

display(queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in ranked queue: 53201
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,action,reason_code,impressions,clicks,ctr,avg_position,position_bucket,bucket_avg_ctr,ctr_gap
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,0.038231,REVIEW_CTR,high_volume_low_ctr,203401.0,2.0,0.000010,0.259483,1-3,0.003138,0.003128
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.038202,REVIEW_CTR,high_volume_low_ctr,193954.0,0.0,0.000000,0.070336,1-3,0.003138,0.003138
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0.038167,REVIEW_CTR,high_volume_low_ctr,195648.0,1.0,0.000005,0.009865,1-3,0.003138,0.003133
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,0.036825,REVIEW_CTR,high_volume_low_ctr,125035.0,0.0,0.000000,2.308282,1-3,0.003138,0.003138
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,0.035370,REVIEW_CTR,high_volume_low_ctr,92128.0,4.0,0.000043,0.039630,1-3,0.003138,0.003094
5,6,client_23a62021009f63c4,content_44f34c0a90047651,0.034157,REVIEW_CTR,high_volume_low_ctr,90223.0,13.0,0.000144,0.518504,1-3,0.003138,0.002994
6,7,client_3197e6291363b4db,content_22588e765b93dfac,0.033580,REVIEW_CTR,high_volume_low_ctr,54938.0,2.0,0.000036,7.162747,4-10,0.003113,0.003077
7,8,client_73cda7b4e4f265ea,content_34e3f342bfd1dbf4,0.031818,REVIEW_CTR,high_volume_low_ctr,59434.0,13.0,0.000219,7.283457,4-10,0.003113,0.002894
8,9,client_861cdcccf8049915,content_c406f6bcaac8a477,0.031812,REVIEW_CTR,high_volume_low_ctr,30545.0,1.0,0.000033,4.225831,4-10,0.003113,0.003080
9,10,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,0.031248,REVIEW_CTR,high_volume_low_ctr,51000.0,13.0,0.000255,1.287686,1-3,0.003138,0.002883


## 2. Build the ranked queue

The baseline score is intentionally transparent:

**score = normalized impressions × CTR gap**

A page receives a higher score when it has substantial search visibility and its CTR is below the average CTR for pages in the same position bucket.

Every scored page receives one reason code (`high_volume_low_ctr`) and one action label (`REVIEW_CTR`).

Only February 2026 measurements are used to construct the score. No March outcome data or future-window information is used.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 3: Top-20 review

top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "High search exposure with CTR below the average for its position bucket."
)

top20["what_would_make_it_wrong"] = (
    "SERP features, search intent, brand effects, or measurement issues "
    "could explain the low CTR instead of a content problem."
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "score",
    "impressions",
    "ctr",
    "avg_position",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,action,reason_code,score,impressions,ctr,avg_position,confidence_note,what_would_make_it_wrong
0,1,REVIEW_CTR,high_volume_low_ctr,0.038231,203401.0,0.000010,0.259483,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
1,2,REVIEW_CTR,high_volume_low_ctr,0.038202,193954.0,0.000000,0.070336,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
2,3,REVIEW_CTR,high_volume_low_ctr,0.038167,195648.0,0.000005,0.009865,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
3,4,REVIEW_CTR,high_volume_low_ctr,0.036825,125035.0,0.000000,2.308282,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
4,5,REVIEW_CTR,high_volume_low_ctr,0.035370,92128.0,0.000043,0.039630,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
5,6,REVIEW_CTR,high_volume_low_ctr,0.034157,90223.0,0.000144,0.518504,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
6,7,REVIEW_CTR,high_volume_low_ctr,0.033580,54938.0,0.000036,7.162747,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
7,8,REVIEW_CTR,high_volume_low_ctr,0.031818,59434.0,0.000219,7.283457,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
8,9,REVIEW_CTR,high_volume_low_ctr,0.031812,30545.0,0.000033,4.225831,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."
9,10,REVIEW_CTR,high_volume_low_ctr,0.031248,51000.0,0.000255,1.287686,High search exposure with CTR below the averag...,"SERP features, search intent, brand effects, o..."


## 3. Top-20 review

The following review checks the highest-ranked pages from the baseline.

For each page, the action and reason code come directly from the rule. The confidence note explains why the signal is useful, while the final note identifies what could make the recommendation wrong.

These are decision-support recommendations, not proof that a page has a CTR problem.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 3: Top-20 review

# SECTION 4: Weak picks + leakage check

print("LEAKAGE CHECK")
print("Feature window used: February 2026")
print("March outcome fields used in score:", False)
print("June sealed month used:", False)
print("Future-window fields used:", False)

print("\nWeak-pick review:")

weak_picks = queue.tail(5)[
    [
        "rank",
        "score",
        "impressions",
        "ctr",
        "avg_position",
        "reason_code"
    ]
]

display(weak_picks)

print(
    "\nWhy these can be weak picks: "
    "the rule only uses search volume and CTR relative to position. "
    "It does not know query intent, SERP features, brand effects, "
    "or content quality."
)

LEAKAGE CHECK
Feature window used: February 2026
March outcome fields used in score: False
June sealed month used: False
Future-window fields used: False

Weak-pick review:


,rank,score,impressions,ctr,avg_position,reason_code
53196,53197,7.552056e-06,2110.0,0.002370,14.880095,high_volume_low_ctr
53197,53198,6.927704e-06,734.0,0.001362,21.106267,high_volume_low_ctr
53198,53199,5.966112e-06,422.0,0.002370,10.902844,high_volume_low_ctr
53199,53200,2.878320e-06,2570.0,0.003113,4.725292,high_volume_low_ctr
53200,53201,1.721591e-07,4497.0,0.003113,7.048699,high_volume_low_ctr



Why these can be weak picks: the rule only uses search volume and CTR relative to position. It does not know query intent, SERP features, brand effects, or content quality.


## 4. Weak picks + leakage check

The baseline can produce false positives because low CTR does not necessarily mean that content needs improvement. Search intent, SERP features, brand effects, seasonality, and measurement availability can all affect CTR.

The weakest-looking picks are therefore treated as review candidates rather than confirmed problems.

The score uses only the February 2026 feature window. March 2026 is not used to calculate the score, and the sealed June 2026 month is not used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.